# Lesson 22 — Iterators & Generators

How for-loops really work, building your own iterators, and memory-efficient generators with yield.

## 🎯 Learning Objectives

- Understand iterables vs iterators
- Use `iter()` and `next()`
- Build a custom iterator class
- Write generator functions with `yield`
- Use generator expressions and `itertools`

## 🔄 Iterable vs Iterator

- **Iterable** — anything you can loop over (list, str, dict, file…). Has `__iter__()`.
- **Iterator** — an object that produces the next value with `__next__()` and remembers its position.

A `for` loop calls `iter()` on the iterable, then `next()` until `StopIteration`.

In [1]:
colors = ["red", "green", "blue"]
it = iter(colors)
print(next(it))
print(next(it))
print(next(it))
try:
    next(it)
except StopIteration:
    print("StopIteration — the iterator is exhausted")

red
green
blue
StopIteration — the iterator is exhausted


## 🛠️ Custom Iterator Class

In [2]:
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value

for n in Countdown(5):
    print(n, end=" ")

5 4 3 2 1 


## ⚡ Generators — `yield`

A **generator function** uses `yield` instead of `return`. It pauses after each `yield` and resumes on the next call — values are produced **one at a time**, on demand.

In [3]:
def countdown(n):
    while n > 0:
        yield n
        n -= 1

gen = countdown(3)
print(gen)
print(next(gen), next(gen), next(gen))
print(list(countdown(5)))

<generator object countdown at 0x7fd95332dd80>
3 2 1
[5, 4, 3, 2, 1]


In [4]:
def show_pauses():
    print("Step 1")
    yield "A"
    print("Step 2")
    yield "B"
    print("Step 3")

for value in show_pauses():
    print("Got:", value)

Step 1
Got: A
Step 2
Got: B
Step 3


### Infinite generator

In [5]:
def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

fib = fibonacci()
print([next(fib) for _ in range(12)])

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89]


### Generator pipeline

In [6]:
def read_lines():
    data = ["10", "x", "25", "", "40", "abc", "5"]
    for line in data:
        yield line

def only_numbers(lines):
    for line in lines:
        if line.isdigit():
            yield int(line)

def doubled(nums):
    for n in nums:
        yield n * 2

print(list(doubled(only_numbers(read_lines()))))

[20, 50, 80, 10]


## 💾 Memory: List vs Generator

In [7]:
import sys
big_list = [x * x for x in range(1_000_000)]
big_gen = (x * x for x in range(1_000_000))
print("List     :", sys.getsizeof(big_list), "bytes")
print("Generator:", sys.getsizeof(big_gen), "bytes")
print("Same sum :", sum(big_list) == sum(big_gen))

List     : 8448728 bytes
Generator: 200 bytes
Same sum : True


## ↪️ `yield from`

In [8]:
def chain(*iterables):
    for it in iterables:
        yield from it

print(list(chain([1, 2], (3, 4), "ab")))

[1, 2, 3, 4, 'a', 'b']


## 🧰 `itertools` Highlights

In [9]:
import itertools as it

print(list(it.islice(it.count(10, 5), 4)))          # 10, 15, 20, 25
print(list(it.islice(it.cycle("AB"), 5)))
print(list(it.chain([1, 2], [3])))
print(list(it.combinations("ABC", 2)))
print(list(it.permutations([1, 2, 3], 2)))
print(list(it.product("ab", [0, 1])))
print(list(it.accumulate([1, 2, 3, 4])))
for key, group in it.groupby("aaabbc"):
    print(key, list(group))

[10, 15, 20, 25]
['A', 'B', 'A', 'B', 'A']
[1, 2, 3]
[('A', 'B'), ('A', 'C'), ('B', 'C')]
[(1, 2), (1, 3), (2, 1), (2, 3), (3, 1), (3, 2)]
[('a', 0), ('a', 1), ('b', 0), ('b', 1)]
[1, 3, 6, 10]
a ['a', 'a', 'a']
b ['b', 'b']
c ['c']


## 📌 Summary

- `for` uses `iter()` + `next()` under the hood.
- Custom iterators implement `__iter__` and `__next__`.
- Generators (`yield`) produce values lazily — perfect for large or infinite data.
- `itertools` provides fast building blocks for iteration.